In [ ]:
# data analysis
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt
from datetime import  datetime  

# data management
from ucimlrepo import fetch_ucirepo
from openmeteopy import OpenMeteo
from openmeteopy.hourly import HourlyHistorical
from openmeteopy.daily import DailyHistorical
from openmeteopy.options import HistoricalOptions
from openmeteopy.utils.constants import *
import pickle

# system 
import os


# zobaczyć 
import zipfile
import io
import requests

# global parameters
name_data_folder = "data"
name_uci_raw = "raw_EP"
name_meteo_raw = 'raw_METEO'

# Part 1 - download data

We download data that contains
- energy usage of `UCL` 
- weather data from UCI Machine Learning Repository
for localization: `Sceaux Ile-de-France` ($48.77^\circ\mathrm{N}, 2.29^\circ\mathrm{E}$)

Methodology
- If data does not exits, download `UCL data` using `ucimlrepo` library, saving it using `pickle` library
- If data exits, load data
- Date, time format: we use `iso8601` (YYYY-MM-DDTHH:mm:ss±hh:mm)

In [ ]:
### ----------- UCI ----------- ###

# flag 
exists = any(filename.startswith(name_uci_raw) for filename in os.listdir(name_data_folder))

# load data
if exists:
    print(f'{name_uci_raw} already exists')
    print(f"{name_uci_raw} Data loading...")
    ## opening file
    with open(f"{name_data_folder}/{name_uci_raw}.pkl", "rb") as file:
        ## loading file
        df_raw_EP = pickle.load(file)
    print(f"Done")
# download data
else:
    print(f"Downloading {name_uci_raw} data ...")
    ## downloading data
    df_raw_EP = fetch_ucirepo(id=235)       
    print(f"Download complete \nSaving data...") 
    ## creating file for writing uci repo
    with open(f"{name_data_folder}/{name_uci_raw}.pkl", 'wb') as file:
        ## saving file
        pickle.dump(df_raw_EP, file)
    print('Done')

# leak check 
num_instances_metadata = df_raw_EP.metadata['num_instances']
num_intances_local = df_raw_EP.data.features.shape[0]
if num_intances_local != num_instances_metadata:
    print(f'''There is leak of data in your data set:
          - global data instances = {num_instances_metadata}
          - local data instances = {num_intances_local}
        Removing local dataset ...
          ''')
    os.remove(f"{name_data_folder}/{name_uci_raw}")
    print('Local dataset removed\nRun cell again')
else:
    print(f"\n---There is no leak in data---\n")

# load UCI data into df
df_EP = df_raw_EP.data.features
# variable information 
print(df_raw_EP.variables)

In [ ]:
### ----------- JOIN DATASETS I ----------- ###

# create final df
## UCI DATA
df_EP_copy = df_EP.copy()
### update `Date` and `Time` column to `datetime` type
df_EP_copy['Date'] = df_EP_copy['Date'] + ' ' + df_EP_copy['Time']
df_EP_copy['Date'] = pd.to_datetime(df_EP_copy['Date'], format='%d/%m/%Y %H:%M:%S', utc=True)
df_EP_copy = df_EP_copy.drop('Time', axis=1)
### set time index
df_EP_copy = df_EP_copy.set_index('Date')
### set float64 data type for rest columns 
df_EP_copy  = df_EP_copy.apply(pd.to_numeric, errors="coerce")
### We need to change Date in out `df` file - we need to get rid off minutes
df_EP_copy = df_EP_copy.resample('H').mean()
df_EP_copy

In [ ]:
### ----------- METEO ----------- ###

# parameters 
longitude = 48.7764 # 48.66 N
latitude =  2.2903 # 2.29 E 
start_date, end_date = str(df_EP_copy.index.date.min()), str(df_EP_copy.index.date.max())

exists = any(filename.startswith(name_meteo_raw) for filename in os.listdir(name_data_folder))

# load data
if exists:
    print(f'{name_meteo_raw} already exists')
    print(f"{name_meteo_raw} Data loading...")
    ## open file
    with open(f"{name_data_folder}/{name_meteo_raw}.pkl", "rb") as file:
        ## load data
        df_raw_meteo = pickle.load(file)
    print(f"Done")
# download data 
else:
    print(f"Downloading {name_meteo_raw} data ...")
    ## set download options
    hourly = HourlyHistorical()
    daily = DailyHistorical()
    options = HistoricalOptions(latitude,
                                longitude, 
                                start_date=start_date,
                                end_date=end_date)
    ## download 
    df_raw_meteo = OpenMeteo(options, hourly.all(), daily.all())
    print(f"Download complete \nSaving data...") 
    ## saving file
    with open(f"{name_data_folder}/{name_meteo_raw}.pkl", "wb") as file:
        pickle.dump(df_raw_meteo, file)
    print('Done')
# Download data
df_meteo = df_raw_meteo.get_pandas()[0]
df_meteo

In [ ]:
### ----------- JOIN DATASETS II ----------- ###

# parameters
name_combined_raw = 'raw_combined'

## METEO DATA
df_meteo_copy = df_meteo.copy()
df_meteo_copy = df_meteo_copy.rename_axis('Date')
df_meteo_copy.index = pd.to_datetime(df_meteo_copy.index, format="%Y-%m-%dT%H:%M", utc=True)
## MERGE DATA
### joining 
df = df_EP_copy
df = df.join(df_meteo_copy, how='outer')
df
### saving data
df.to_json(f"{name_data_folder}/{name_combined_raw}.json")

# Part 2 - data engineering 

In [ ]:
### ----------- Loading data ----------- ###

# parameters

# loading data
df = pd.read_json(f"{name_data_folder}/{name_combined_raw}.json")
df

# TODO - check 
# change format: '%d/%m/%Y %H:%M:%S' -> '%d/%m/%Y' by taking mean per day
# df = df.resample('D').mean()